In [4]:
import pandas as pd, numpy as np, xgboost as xgb, json, mlflow
from sklearn.metrics import average_precision_score, roc_auc_score

feast_feats = ["hour", "day_of_week", "tx_count_before", "secs_since_last",
               "amt_vs_avg", "device_changed", "addr_changed", "email_changed"]
msg_feats = (["TransactionAmt", "card1", "card2", "card3", "card5", "addr1", "addr2", "dist1"]
             + [f"C{i}" for i in range(1, 15)] + [f"D{i}" for i in range(1, 16)])
all_feats = feast_feats + msg_feats

f = pd.read_parquet("data/features.parquet")
raw = pd.read_csv("data/train_transaction.csv", usecols=["TransactionID"] + msg_feats)
df = f.merge(raw, on="TransactionID").sort_values("event_timestamp").reset_index(drop=True)

n = len(df); a, b = int(n * 0.72), int(n * 0.80)
X, y = df[all_feats], df["isFraud"]
ratio = (y[:a] == 0).sum() / (y[:a] == 1).sum()

m = xgb.XGBClassifier(n_estimators=1000, max_depth=6, learning_rate=0.05, subsample=0.8,
                      colsample_bytree=0.8, scale_pos_weight=ratio, tree_method="hist",
                      eval_metric="aucpr", early_stopping_rounds=50, random_state=42)
m.fit(X[:a], y[:a], eval_set=[(X[a:b], y[a:b])], verbose=100)

p = m.predict_proba(X[b:])[:, 1]
pr, roc = average_precision_score(y[b:], p), roc_auc_score(y[b:], p)
print("serving model  PR-AUC:", round(pr, 4), " ROC-AUC:", round(roc, 4), " rounds:", m.best_iteration)

m.save_model("models/xgb_serving.json")
json.dump({"feast": feast_feats, "message": msg_feats}, open("models/serving_features.json", "w"))

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("fraud-detection")
with mlflow.start_run(run_name="xgb_serving"):
    mlflow.log_metrics({"pr_auc": pr, "roc_auc": roc})

[0]	validation_0-aucpr:0.35225
[100]	validation_0-aucpr:0.49017
[200]	validation_0-aucpr:0.50812
[300]	validation_0-aucpr:0.52337
[400]	validation_0-aucpr:0.53360
[500]	validation_0-aucpr:0.54283
[600]	validation_0-aucpr:0.54916
[700]	validation_0-aucpr:0.55179
[800]	validation_0-aucpr:0.55506
[900]	validation_0-aucpr:0.55732
[999]	validation_0-aucpr:0.56073
serving model  PR-AUC: 0.4584  ROC-AUC: 0.8808  rounds: 993


In [5]:
%%writefile src/producer.py
import json, time, sys
import pandas as pd
from kafka import KafkaProducer

RATE = float(sys.argv[1]) if len(sys.argv) > 1 else 5
LIMIT = int(sys.argv[2]) if len(sys.argv) > 2 else 200

msg_feats = (["TransactionAmt", "card1", "card2", "card3", "card5", "addr1", "addr2", "dist1"]
             + [f"C{i}" for i in range(1, 15)] + [f"D{i}" for i in range(1, 16)])

f = pd.read_parquet("data/features.parquet", columns=["TransactionID", "card_id", "event_timestamp", "isFraud"])
raw = pd.read_csv("data/train_transaction.csv", usecols=["TransactionID"] + msg_feats)
df = f.merge(raw, on="TransactionID").sort_values("event_timestamp")
df = df.iloc[int(len(df) * 0.8):].head(LIMIT)

producer = KafkaProducer(bootstrap_servers="localhost:9092",
                         value_serializer=lambda v: json.dumps(v).encode("utf-8"))

for _, r in df.iterrows():
    msg = {"transaction_id": int(r.TransactionID), "card_id": int(r.card_id),
           "true_label": int(r.isFraud)}
    for c in msg_feats:
        msg[c] = None if pd.isna(r[c]) else float(r[c])
    producer.send("transactions", msg)
    time.sleep(1 / RATE)

producer.flush()
print("sent", len(df), "messages")

Overwriting src/producer.py


In [6]:
import pandas as pd
f = pd.read_parquet("data/features.parquet", columns=["event_timestamp"]).sort_values("event_timestamp")
cut = f["event_timestamp"].iloc[int(len(f) * 0.8)]
print(cut.strftime("%Y-%m-%dT%H:%M:%S"))

2018-04-21T02:55:00


In [7]:
%%writefile src/api.py
import json, time
import pandas as pd
import xgboost as xgb
from fastapi import FastAPI, Body
from feast import FeatureStore

cols = json.load(open("models/serving_features.json"))
FEAST_COLS, MSG_COLS = cols["feast"], cols["message"]
ALL = FEAST_COLS + MSG_COLS
THRESHOLD = 0.5   # alert if score is at or above this; we tune it later

model = xgb.XGBClassifier()
model.load_model("models/xgb_serving.json")
store = FeatureStore(repo_path="feature_repo")
app = FastAPI(title="Fraud scoring service")


def score_one(tx: dict) -> dict:
    t0 = time.perf_counter()
    res = store.get_online_features(
        features=[f"card_features:{c}" for c in FEAST_COLS],
        entity_rows=[{"card_id": int(tx["card_id"])}],
    ).to_dict()
    row = {c: res[c][0] for c in FEAST_COLS}
    for c in MSG_COLS:
        row[c] = tx.get(c)
    X = pd.DataFrame([row])[ALL].astype("float64")
    p = float(model.predict_proba(X)[0, 1])
    return {
        "transaction_id": tx.get("transaction_id"),
        "fraud_score": round(p, 4),
        "alert": p >= THRESHOLD,
        "latency_ms": round((time.perf_counter() - t0) * 1000, 1),
    }


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/score")
def score(tx: dict = Body(...)):
    return score_one(tx)

Writing src/api.py


In [8]:
%%writefile src/test_api.py
import pandas as pd, requests, json

cols = json.load(open("models/serving_features.json"))["message"]
f = pd.read_parquet("data/features.parquet", columns=["TransactionID", "card_id", "event_timestamp", "isFraud"])
raw = pd.read_csv("data/train_transaction.csv", usecols=["TransactionID"] + cols)
df = f.merge(raw, on="TransactionID").sort_values("event_timestamp")
test = df.iloc[int(len(df) * 0.8):]
sample = pd.concat([test[test.isFraud == 1].head(5), test[test.isFraud == 0].head(5)])

for _, r in sample.iterrows():
    msg = {"transaction_id": int(r.TransactionID), "card_id": int(r.card_id)}
    for c in cols:
        msg[c] = None if pd.isna(r[c]) else float(r[c])
    out = requests.post("http://127.0.0.1:8000/score", json=msg).json()
    print("true:", int(r.isFraud), "| score:", out["fraud_score"], "| alert:", out["alert"], "|", out["latency_ms"], "ms")

Writing src/test_api.py


In [9]:
%%writefile src/consumer.py
import json, sqlite3, time
import requests
from kafka import KafkaConsumer

DB = "data/scores.db"
API = "http://127.0.0.1:8000/score"

con = sqlite3.connect(DB)
con.execute("""CREATE TABLE IF NOT EXISTS scores (
    transaction_id INTEGER PRIMARY KEY,
    card_id INTEGER,
    fraud_score REAL,
    alert INTEGER,
    latency_ms REAL,
    true_label INTEGER,
    scored_at REAL,
    status TEXT DEFAULT 'new',
    analyst_label INTEGER
)""")
con.commit()

consumer = KafkaConsumer(
    "transactions", bootstrap_servers="localhost:9092",
    auto_offset_reset="earliest", consumer_timeout_ms=30000,
    value_deserializer=lambda v: json.loads(v.decode("utf-8")))

n = 0
for m in consumer:
    tx = m.value
    out = requests.post(API, json=tx, timeout=10).json()
    con.execute(
        "INSERT OR REPLACE INTO scores (transaction_id, card_id, fraud_score, alert, latency_ms, true_label, scored_at) "
        "VALUES (?,?,?,?,?,?,?)",
        (tx["transaction_id"], tx["card_id"], out["fraud_score"], int(out["alert"]),
         out["latency_ms"], tx.get("true_label"), time.time()))
    con.commit()
    n += 1
    if n % 20 == 0:
        print("scored", n)

print("done, scored", n, "messages")

Writing src/consumer.py


In [10]:
import sqlite3, pandas as pd
s = pd.read_sql("SELECT * FROM scores", sqlite3.connect("data/scores.db"))
print(len(s), "scored |", int(s.alert.sum()), "alerts |", int(s.true_label.sum()), "real frauds")
print("avg latency ms:", round(s.latency_ms.mean(), 1), "| max:", round(s.latency_ms.max(), 1))
print("avg score, fraud rows:", round(s[s.true_label == 1].fraud_score.mean(), 3),
      "| normal rows:", round(s[s.true_label == 0].fraud_score.mean(), 3))
print(s.sort_values("fraud_score", ascending=False).head(10)[["transaction_id", "fraud_score", "alert", "true_label"]])

200 scored | 23 alerts | 13 real frauds
avg latency ms: 13.7 | max: 97.7
avg score, fraud rows: 0.467 | normal rows: 0.145
     transaction_id  fraud_score  alert  true_label
78          3459510       0.9970      1           1
122         3459554       0.8933      1           0
2           3459434       0.8747      1           0
143         3459575       0.8203      1           0
145         3459577       0.7932      1           0
3           3459435       0.7850      1           0
115         3459547       0.7672      1           0
139         3459571       0.7667      1           0
149         3459581       0.7123      1           0
148         3459580       0.7059      1           0


In [11]:
%%writefile src/check_drift.py
import json, shutil
import numpy as np, pandas as pd, xgboost as xgb, mlflow
from scipy.stats import ks_2samp
from sklearn.metrics import average_precision_score

cols = json.load(open("models/serving_features.json"))
FEATS = cols["feast"] + cols["message"]
DRIFT_STAT = 0.10      # a feature "drifted" if its distribution moved by more than this (KS statistic)
TRIGGER_SHARE = 0.30   # retrain if more than 30% of features drifted

f = pd.read_parquet("data/features.parquet")
raw = pd.read_csv("data/train_transaction.csv", usecols=["TransactionID"] + cols["message"])
df = f.merge(raw, on="TransactionID").sort_values("event_timestamp").reset_index(drop=True)
n = len(df); a, b = int(n * 0.72), int(n * 0.80)
ref, cur = df.iloc[:a], df.iloc[b:]          # reference = training period, current = newest period

# 1) drift per feature
stats = {}
for c in FEATS:
    x, y = ref[c].dropna(), cur[c].dropna()
    stats[c] = ks_2samp(x, y).statistic if len(x) > 100 and len(y) > 100 else 0.0
drifted = [c for c, s in stats.items() if s > DRIFT_STAT]
share = len(drifted) / len(FEATS)
print(f"drifted features: {len(drifted)}/{len(FEATS)} ({share:.0%})")
print("top 5:", sorted(stats.items(), key=lambda kv: -kv[1])[:5])

# 2) Evidently HTML report (optional)
try:
    from evidently import Report, Dataset, DataDefinition
    from evidently.presets import DataDriftPreset
    dd = DataDefinition(numerical_columns=FEATS)
    r = ref[FEATS].sample(20000, random_state=1)
    c = cur[FEATS].sample(20000, random_state=1)
    snap = Report([DataDriftPreset()]).run(Dataset.from_pandas(c, data_definition=dd),
                                           Dataset.from_pandas(r, data_definition=dd))
    snap.save_html("docs/drift_report.html")
    print("saved docs/drift_report.html")
except Exception as e:
    print("Evidently report skipped:", e)

# 3) retraining trigger
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("fraud-detection")
if share <= TRIGGER_SHARE:
    print("No retraining needed.")
else:
    print("Drift above threshold -> retraining a challenger model")
    cut = int(b * 0.9)
    tr, va = df.iloc[:cut], df.iloc[cut:b]         # train on everything before the newest period
    ratio = (tr.isFraud == 0).sum() / (tr.isFraud == 1).sum()
    cand = xgb.XGBClassifier(n_estimators=1000, max_depth=6, learning_rate=0.05, subsample=0.8,
                             colsample_bytree=0.8, scale_pos_weight=ratio, tree_method="hist",
                             eval_metric="aucpr", early_stopping_rounds=50, random_state=42)
    cand.fit(tr[FEATS], tr.isFraud, eval_set=[(va[FEATS], va.isFraud)], verbose=False)

    champ = xgb.XGBClassifier(); champ.load_model("models/xgb_serving.json")
    pr_c = average_precision_score(cur.isFraud, cand.predict_proba(cur[FEATS])[:, 1])
    pr_o = average_precision_score(cur.isFraud, champ.predict_proba(cur[FEATS])[:, 1])
    print(f"current model PR-AUC {pr_o:.4f} | challenger PR-AUC {pr_c:.4f}")

    with mlflow.start_run(run_name="auto_retrain"):
        mlflow.log_metrics({"drift_share": share, "champion_pr": pr_o, "challenger_pr": pr_c})
        if pr_c > pr_o:
            shutil.copy("models/xgb_serving.json", "models/xgb_serving_prev.json")
            cand.save_model("models/xgb_serving.json")
            mlflow.log_param("promoted", True)
            print("Challenger is better -> promoted. Restart the API to load it.")
        else:
            mlflow.log_param("promoted", False)
            print("Challenger is not better -> keeping the current model.")

Writing src/check_drift.py


In [12]:
%%writefile src/dashboard.py
import sqlite3
import pandas as pd
import streamlit as st

DB = "data/scores.db"
st.set_page_config(page_title="Fraud Analyst Dashboard", layout="wide")
st.title("Fraud Analyst Dashboard")


def q(sql, params=()):
    con = sqlite3.connect(DB, timeout=10)
    try:
        return pd.read_sql(sql, con, params=params)
    finally:
        con.close()


def run(sql, params=()):
    con = sqlite3.connect(DB, timeout=10)
    try:
        con.execute(sql, params)
        con.commit()
    finally:
        con.close()


try:
    s = q("SELECT * FROM scores")
except Exception as e:
    st.error(f"Cannot read {DB}. Run the producer and consumer first. ({e})")
    st.stop()

threshold = st.sidebar.slider("Alert threshold", 0.05, 0.95, 0.50, 0.05)
st.sidebar.caption("Transactions with a score at or above this value are alerts.")

alerts = s[s.fraud_score >= threshold]
reviewed = alerts[alerts.status != "new"]
confirmed = (reviewed.status == "confirmed_fraud").sum()

c1, c2, c3, c4, c5 = st.columns(5)
c1.metric("Transactions scored", len(s))
c2.metric("Alerts", len(alerts))
c3.metric("Open cases", int((alerts.status == "new").sum()))
c4.metric("Avg latency (ms)", round(s.latency_ms.mean(), 1) if len(s) else 0)
c5.metric("Analyst precision", f"{confirmed / len(reviewed):.0%}" if len(reviewed) else "n/a")

st.subheader("Cases to review (highest risk first)")
view = st.radio("Show", ["Open", "All alerts"], horizontal=True)
cases = alerts if view == "All alerts" else alerts[alerts.status == "new"]
cases = cases.sort_values("fraud_score", ascending=False)
st.dataframe(
    cases[["transaction_id", "card_id", "fraud_score", "status", "latency_ms", "true_label"]]
    .rename(columns={"true_label": "true_label (demo only)"}),
    use_container_width=True, height=300)

st.subheader("Review a case")
if len(cases):
    tid = st.selectbox("Transaction ID", cases.transaction_id.tolist())
    row = s[s.transaction_id == tid].iloc[0]
    st.write(f"Score **{row.fraud_score:.3f}** | card **{int(row.card_id)}** | "
             f"current status **{row.status}**")
    b1, b2, b3 = st.columns(3)
    for col, label, status, lab in [
        (b1, "Confirm fraud", "confirmed_fraud", 1),
        (b2, "False alarm", "false_alarm", 0),
        (b3, "Escalate", "escalated", None),
    ]:
        if col.button(label, key=f"{label}_{tid}"):
            run("UPDATE scores SET status=?, analyst_label=? WHERE transaction_id=?",
                (status, lab, int(tid)))
            st.rerun()
else:
    st.info("No cases to show. Lower the threshold or send more transactions.")

st.subheader("Feedback labels")
fb = q("SELECT transaction_id, card_id, fraud_score, status, analyst_label "
       "FROM scores WHERE analyst_label IS NOT NULL")
st.write(f"{len(fb)} labelled by analysts")
st.download_button("Download feedback labels (CSV)", fb.to_csv(index=False), "feedback_labels.csv")

Writing src/dashboard.py
